In [138]:
import fsspec
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import multiprocessing
import numpy as np
import pandas as pd
import xarray as xr

In [139]:
local = fsspec.filesystem('file')

In [140]:
files = local.glob('C:/Users\Ian\projects\python3-crlx\__scratch__/test_data/*PCO2*.nc')

In [141]:
with multiprocessing.Pool() as pool:
    ds_list = pool.map(xr.open_dataset,files)

In [142]:
ds = xr.combine_by_coords(ds_list)

In [143]:
figsize = (6,5)
fig = plt.figure(figsize = figsize, constrained_layout = True)
grid = (3,1)
ax1 = plt.subplot2grid(grid, (0,0), colspan = 1,rowspan = 1)
ax2 = plt.subplot2grid(grid, (1,0), colspan = 1,rowspan = 1)
ax3 = plt.subplot2grid(grid, (2,0), colspan = 1,rowspan = 1)


ax1.plot(ds.time, ds.pco2)
ax1.set_title('Not Separated (Includes Standard, Water, and Air Readings)')
ax1.set_ylabel(r'pCO2 (${\mu}atm$)')
ax1.set_ylim(0,550)
ax1.set_xlim(ds.time.min(), ds.time.max())
ax1.xaxis.set_major_locator(mdates.DayLocator(interval = 1))
ax1.xaxis.set_minor_locator(mdates.HourLocator(interval = 1))
ax1.yaxis.set_major_locator(mticker.MultipleLocator(100))



In [144]:
pco2 = ds.pco2

In [145]:
pco2

In [146]:
np.argmin(pco2.values == 0)

In [147]:
blank_times = pd.to_datetime(pco2.where(pco2 < 5, drop = True).time.values).tolist()


In [148]:
blank_list = []
std1_list = []
std2_list = []
std3_list = []
std4_list = []
std5_list = []
pco2w_list = []
pco2a_list = []

for bt in blank_times:
    idx = blank_times.index(bt)
    if idx == 0:
        _pco2 = pco2.sel(time = slice(None, bt))
        larr = len(_pco2)
        if larr == 0:
            continue
        blank = _pco2[-1]
        if larr == 1:
            blank_list.append(blank)
            continue
        if larr < 6:
            a = _pco2[-larr:-1]
            pco2a_list.append(a)
            continue
        else:
            a = _pco2[-6:-1]
        if 6 < larr < 36:
            w = _pco2[-larr:-6]
        else:
            w = _pco2[-36:-6]
        try:
            std5 = _pco2[-37]
            std4 = _pco2[-38]
            std3 = _pco2[-39]
            std2 = _pco2[-40]
            std1 = _pco2[-41]
        except:
            continue
        
        try:
            blank_list.append(blank)
            pco2a_list.append(w)
            pco2a_list.append(a)
            std5_list.append(std5)
            std4_list.append(std4)
            std3_list.append(std3)
            std2_list.append(std2)
            std1_list.append(std1)
        except:
            continue
    
    elif idx != len(blank_times)-1:
        _pco2 = pco2.sel(time = slice(bt, blank_times[idx + 1]))
        blank2 = _pco2[-1]
        a = _pco2[-6:-1]
        w = _pco2[-36:-6]
        std5 = _pco2[-37]
        std4 = _pco2[-38]
        std3 = _pco2[-39]
        std2 = _pco2[-40]
        std1 = _pco2[-41]
        blank1 = _pco2[-42]
        blank_list.append(blank2)
        blank_list.append(blank1)
        std1_list.append(std1)
        std2_list.append(std2)
        std3_list.append(std3)
        std4_list.append(std4)
        std5_list.append(std5)
        pco2w_list.append(w)
        pco2a_list.append(a)
        
    
        
        
        

In [149]:
pco2w = xr.combine_by_coords(pco2w_list).drop_duplicates('time')
pco2a = xr.combine_by_coords(pco2a_list).drop_duplicates('time')

blank = xr.concat(blank_list,dim = 'time').drop_duplicates('time')
std1 = xr.concat(std1_list,dim = 'time').drop_duplicates('time')
std2 = xr.concat(std2_list,dim = 'time').drop_duplicates('time')
std3 = xr.concat(std3_list,dim = 'time').drop_duplicates('time')
std4 = xr.concat(std4_list,dim = 'time').drop_duplicates('time')
std5 = xr.concat(std5_list,dim = 'time').drop_duplicates('time')

In [150]:
figsize = (6,5)
fig = plt.figure(figsize = figsize, constrained_layout = True)
grid = (4,1)
ax1 = plt.subplot2grid(grid, (0,0), colspan = 1,rowspan = 1)
ax2 = plt.subplot2grid(grid, (1,0), colspan = 1,rowspan = 1)
ax3 = plt.subplot2grid(grid, (2,0), colspan = 1,rowspan = 1)
ax4 = plt.subplot2grid(grid, (3,0), colspan = 1,rowspan = 1)


ax1.plot(ds.time, ds.pco2)
ax1.set_title('Not Separated (Includes Standard, Water, and Air Readings)')
ax1.set_ylabel(r'pCO2 (${\mu}atm$)')
ax1.set_ylim(0,550)
ax1.set_xlim(ds.time.min(), ds.time.max())
ax1.xaxis.set_major_locator(mdates.DayLocator(interval = 1))
ax1.xaxis.set_minor_locator(mdates.HourLocator(interval = 1))
ax1.yaxis.set_major_locator(mticker.MultipleLocator(100))
ax1.yaxis.set_minor_locator(mticker.MultipleLocator(25))

ax2.plot(pco2w.time, pco2w.pco2)
ax2.set_xlim(ds.time.min(), ds.time.max())
ax2.set_ylim(200,550)
ax2.xaxis.set_major_locator(mdates.DayLocator(interval = 1))
ax2.xaxis.set_minor_locator(mdates.HourLocator(interval = 1))
ax2.yaxis.set_major_locator(mticker.MultipleLocator(100))
ax2.yaxis.set_minor_locator(mticker.MultipleLocator(25))
ax2.set_title('pco2w')
ax2.set_ylabel(r'pCO2 (${\mu}atm$)')

ax3.plot(pco2a.time, pco2a.pco2)
ax3.set_xlim(ds.time.min(), ds.time.max())
ax3.set_ylim(410,425)
ax3.xaxis.set_major_locator(mdates.DayLocator(interval = 1))
ax3.xaxis.set_minor_locator(mdates.HourLocator(interval = 1))
ax3.yaxis.set_major_locator(mticker.MultipleLocator(10))
ax3.yaxis.set_minor_locator(mticker.MultipleLocator(1))
ax3.set_title('pco2a')
ax3.set_ylabel(r'pCO2 (${\mu}atm$)')

ax4.plot(blank.time, blank, label = 'Blank')
ax4.plot(std1.time, std1, label = 'Standard 1')
ax4.plot(std2.time, std2, label = 'Standard 2')
ax4.plot(std3.time, std3, label = 'Standard 3')
ax4.plot(std3.time, std4, label = 'Standard 4')
ax4.plot(std5.time, std5, label = 'Standard 5')
ax4.set_xlim(ds.time.min(), ds.time.max())
ax4.set_ylim(0,550)
ax4.xaxis.set_major_locator(mdates.DayLocator(interval = 1))
ax4.xaxis.set_minor_locator(mdates.HourLocator(interval = 1))
ax4.yaxis.set_major_locator(mticker.MultipleLocator(100))
ax4.yaxis.set_minor_locator(mticker.MultipleLocator(25))
ax4.set_title('Blank and Standards')
ax4.set_ylabel(r'pCO2 (${\mu}atm$)')

plt.savefig('pco2_split.jpg',dpi = 600)